In [2]:
import pandas as pd

# 1. Load the dataset using the Google Colab file path
file_path = '/content/sample_data/hotel_bookings.csv'
df = pd.read_csv(file_path)

# 2. Smoke Test 1: How big is the system we are testing?
print(f"Dataset Shape: {df.shape[0]} rows and {df.shape[1]} columns.\n")

# 3. Smoke Test 2: Where are the missing values (data bugs)?
print("Missing Data Report (Number of missing values per column):")
missing_data = df.isnull().sum()

# Only print the columns that actually have missing data
print(missing_data[missing_data > 0])

Dataset Shape: 119390 rows and 32 columns.

Missing Data Report (Number of missing values per column):
children         4
country        488
agent        16340
company     112593
dtype: int64


In [3]:
# 1. Drop the 'company' column entirely (too much missing data)
df = df.drop(columns=['company'])

# 2. Impute (fill in) the missing values for the other columns
df['agent'] = df['agent'].fillna(0)
df['country'] = df['country'].fillna('Unknown')
df['children'] = df['children'].fillna(0)

# 3. Regression Test: Verify there are no more missing values
print("Post-Fix Data Report:")
missing_data_after = df.isnull().sum()
remaining_bugs = missing_data_after[missing_data_after > 0]

if remaining_bugs.empty:
    print("✅ All missing data bugs have been successfully resolved!")
else:
    print("⚠️ Still missing data in:")
    print(remaining_bugs)

# 4. Check our new shape
print(f"New Dataset Shape: {df.shape[0]} rows and {df.shape[1]} columns.")

Post-Fix Data Report:
✅ All missing data bugs have been successfully resolved!
New Dataset Shape: 119390 rows and 31 columns.


In [4]:
# 1. Create helper columns to calculate totals
df['total_guests'] = df['adults'] + df['children'] + df['babies']
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']

# 2. Run the tests to find the logical bugs
ghost_bookings = df[df['total_guests'] == 0]
zero_night_stays = df[df['total_nights'] == 0]

# 3. Print the Bug Report
print("--- LOGICAL BUG REPORT ---")
print(f"Ghost Bookings (0 guests): {len(ghost_bookings)} rows")
print(f"Zero-Night Stays (0 nights): {len(zero_night_stays)} rows")

# Let's peek at one of the ghost bookings to see how weird it is
if len(ghost_bookings) > 0:
    print("\nSample Ghost Booking:")
    print(ghost_bookings[['adults', 'children', 'babies', 'market_segment']].head(1))

--- LOGICAL BUG REPORT ---
Ghost Bookings (0 guests): 180 rows
Zero-Night Stays (0 nights): 715 rows

Sample Ghost Booking:
      adults  children  babies market_segment
2224       0       0.0       0      Corporate


In [5]:
# 1. Filter the dataset to keep only logical, valid bookings
df_clean = df[(df['total_guests'] > 0) & (df['total_nights'] > 0)]

# 2. QA Check: Verify exactly how much bad data we removed
dropped_rows = len(df) - len(df_clean)
print(f"Removed {dropped_rows} illogical rows.")
print(f"Final Pristine Dataset Shape: {df_clean.shape[0]} rows and {df_clean.shape[1]} columns.\n")

# 3. Phase 3: Export the Final Product
output_path = '/content/hotel_bookings_CLEAN.csv'
df_clean.to_csv(output_path, index=False)
print(f"✅ Success! Clean dataset safely exported to: {output_path}")

Removed 825 illogical rows.
Final Pristine Dataset Shape: 118565 rows and 33 columns.

✅ Success! Clean dataset safely exported to: /content/hotel_bookings_CLEAN.csv
